# BayMorph — Recovering a Simulated Galaxy

This is the shortest complete BayMorph analysis. BayMorph generates a noisy
single-Sérsic galaxy with known parameters, fits it with the production
Bayesian workflow, and compares the posterior with the known truth.

Run the cells from top to bottom. The example is intentionally small enough
for interactive use.


In [ ]:
from pathlib import Path

import numpy as np
from astropy.table import Table

from baymorph import BayMorphRunner
from baymorph.plotting import plot_log_image


config_path = Path("configs/simulated_galaxy.yaml")


## 1. Run the complete analysis

The YAML file records the simulation truth, observational setup, scientific
priors, likelihood, and sampler settings. The runner generates the synthetic
FITS image, prepares it, performs Bayesian inference, and writes the
diagnostics.


In [ ]:
runner = BayMorphRunner(config_path)
fit = runner.run(diagnostics=True)


## 2. Inspect the simulated observation

This is the noisy image presented to the same fitting machinery used for
observed data.


In [ ]:
plot_log_image(
    runner.simulation.image,
    title="Simulated single-Sérsic observation",
    colorbar_label="Counts",
    show=True,
)


## 3. Check parameter recovery

The simulation truth was fixed before inference. The table places those known
values beside the weighted posterior summaries returned by BayMorph. Angular
values are expressed in radians to match the fitted parameterization.


In [ ]:
truth = dict(
    runner.config["simulation"]["truth"]
)
truth["theta"] = np.deg2rad(
    truth.pop("theta_deg")
)

parameters = (
    "flux",
    "re",
    "n",
    "q",
    "theta",
    "x0",
    "y0",
)

recovery_rows = []

for parameter in parameters:
    recovery_rows.append(
        {
            "parameter": parameter,
            "truth": truth[parameter],
            **fit.posterior.summary(parameter),
        }
    )

recovery_table = Table(
    rows=recovery_rows
)
recovery_table


## 4. Inspect the fit summary

The returned fit object contains the posterior samples, Bayesian evidence,
goodness-of-fit statistics, best-fitting parameters, and paths to the
persisted diagnostic products.


In [ ]:
fit.summary()
